```sh
pip install sagemaker-core ipywidgets
```

### Notes

- Sagemaker container available support until python 3.12
- Referencia: https://github.com/aws/deep-learning-containers/blob/master/available_images.md

- Instance types to test: ml.g4dn.xlarge, ml.g6.xlarge, ml.trn1.2xlarge 
- https://github.com/aws/amazon-sagemaker-examples/blob/default/%20%20%20%20%20%20%20%20%20sagemaker-core/introductory-blog-sagemaker-core/introductory-blog-sagemaker-core.ipynb

In [1]:
from datetime import datetime, timezone
from sagemaker.core.helper.session_helper import Session, s3_path_join

# Set up region, role and bucket parameters used throughout the notebook.
sagemaker_session = Session(
    default_bucket_prefix='congestion-bogota-training'
)
# region = sagemaker_session.boto_region_name
# role = get_execution_role()
region = 'us-east-1'
role = 'arn:aws:iam::195419001736:role/SagemakerExecutionRole'
bucket_name = "sagemaker-congestion-bogota"

print(f"AWS region: {region}")
print(f"Execution role: {role}")
print(f"Default S3 bucket: {bucket_name}")

[01/22/26 21:49:12] INFO     NumExpr defaulting to 8 threads.                                          ]8;id=868658;file:///home/jose_marquez/miniconda3/envs/traffic-gnn-v2/lib/python3.10/site-packages/numexpr/utils.py\utils.py]8;;\:]8;id=322931;file:///home/jose_marquez/miniconda3/envs/traffic-gnn-v2/lib/python3.10/site-packages/numexpr/utils.py#164\164]8;;\

[01/22/26 21:49:13] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=668241;file:///home/jose_marquez/miniconda3/envs/traffic-gnn-v2/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=133515;file:///home/jose_marquez/miniconda3/envs/traffic-gnn-v2/lib/python3.10/site-packages/botocore/credentials.py#1392\1392]8;;\

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/jose_marquez/.config/sagemaker/config.yaml
AWS region: us-east-1
Execution role: arn:aws:iam::195419001736:role/SagemakerExecutionRole
Default S3 bucket: sagemaker-congestion-bogota


In [2]:
import os
import tempfile
import tarfile
import boto3

TAR_SOURCE_FILENAME = 'sourcedir.tar.gz'


def tar_and_upload_source_code(source_path: str, bucket: str, s3_key_prefix: str='code'):
    '''Compress only the training code and upload to S3.'''

    include_files = [
        'requirements-sagemaker.txt',
    ]

    with tempfile.TemporaryDirectory() as tmp_dir_name:
        tar_file_name = os.path.join(tmp_dir_name, TAR_SOURCE_FILENAME)
        with tarfile.open(tar_file_name, mode='w:gz', dereference=True) as tar:
            src_root = os.path.join(source_path, 'src')
            for root, dirs, files in os.walk(src_root):
                dirs[:] = [d for d in dirs if d != '__pycache__']
                for fname in files:
                    if not fname.endswith('.py'):
                        continue
                    abs_path = os.path.join(root, fname)
                    rel_path = os.path.relpath(abs_path, source_path)
                    tar.add(abs_path, arcname=rel_path)

            for rel_path in include_files:
                abs_path = os.path.join(source_path, rel_path)
                if os.path.exists(abs_path):
                    tar.add(abs_path, arcname=rel_path)

        s3_key = '/'.join([s3_key_prefix, TAR_SOURCE_FILENAME])
        boto3.client('s3').upload_file(tar_file_name, bucket, s3_key)
        return s3_path_join('s3://', bucket, s3_key)

source_code_uri = tar_and_upload_source_code('..', bucket_name, 'congestion-bogota/code')


                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=339423;file:///home/jose_marquez/miniconda3/envs/traffic-gnn-v2/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=348405;file:///home/jose_marquez/miniconda3/envs/traffic-gnn-v2/lib/python3.10/site-packages/botocore/credentials.py#1392\1392]8;;\

In [3]:
image = "195419001736.dkr.ecr.us-east-1.amazonaws.com/congestion-bogota-sagemaker:latest"

job_name = f"congestion-bogota-{datetime.now(timezone.utc):%Y%m%d-%H%M%S}"  # Name of training job
# instance_type = "ml.g5.8xlarge"
instance_type = "ml.g4dn.8xlarge"  # SageMaker instance type to use for training
# instance_type = "ml.g6.8xlarge"

instance_count = 1  # Number of instances to use for training
volume_size_in_gb = 100# Amount of storage to allocate to training job
max_runtime_in_seconds = 36* 60 * 60  # 6 hours
project_s3_path = s3_path_join("s3://", bucket_name, "congestion-bogota")
s3_job_path = s3_path_join(project_s3_path, job_name)

train_data_uri = "s3://ine-projects-incoming-data/congestion-prediction/co/movilidadbogota"
validation_data_uri = "s3://ine-projects-incoming-data/congestion-prediction/co/movilidadbogota"
# train_data_uri = f'{s3_job_path}/data/train'
# validation_data_uri = f'{s3_job_path}/data/validation'
s3_output_path = project_s3_path

# Specify hyperparameters
hyper_parameters = {
    "epochs": "2",
    "batch-size": "16",
    "learning-rate": "0.001",
    "model-type": "GConvLSTM",
    "input-seq-len": "150",
    "output-seq-len": "50",
    "node-batch-size": "500",
    "train-ratio": "0.8",
    "include-weather": "True",
    "include-velocity-target": "True",
    "seeded-batch-sampler": "True",
    "num-workers": "4",
    "drop-last": "False",
    "shuffle-time": "True",
    "seeded-batch-seed": "42",
    "pin-memory": "True",
    "prefetch-factor": "2",
    "persistent-workers": "True",
    "cache-local": "True",
    "cache-dir": "/opt/ml/input/data/cache",
    "use-minimal-datasets": "True",
    "sagemaker_submit_directory": source_code_uri,
    "sagemaker_program": "src/training_entrypoint.py",
    "loss-high-level-threshold": "2",
    "loss-high-level-lambda": "0.5",
    "loss-piecewise-threshold": "4",
    "loss-piecewise-mae-lambda": "0.5",
    "loss-directional-lambda": "0.3",
    "timing-steps": "20",
}


In [4]:
from sagemaker.core.resources import TrainingJob
from sagemaker.core.shapes import (
    AlgorithmSpecification,
    Channel,
    DataSource,
    ResourceConfig,
    StoppingCondition,
    OutputDataConfig,
    MetricDefinition,
    FileSystemDataSource,
)

efs_id = "fs-099e29197c0a3e353"
efs_path = "/data/datos_procesados"

metric_definitions = [
    MetricDefinition(name="train_loss", regex="train_loss=([0-9\\.]+)"),
    MetricDefinition(name="loss_level", regex="loss_level=([0-9\\.]+)"),
    MetricDefinition(name="loss_velocity", regex="loss_velocity=([0-9\\.]+)"),
    MetricDefinition(name="val_loss", regex="val_loss=([0-9\\.]+)"),
    MetricDefinition(name="val_loss_level", regex="val_loss_level=([0-9\\.]+)"),
    MetricDefinition(name="val_loss_velocity", regex="val_loss_velocity=([0-9\\.]+)"),
    MetricDefinition(name="time_data_wait", regex="time_data_wait=([0-9\.]+)"),
    MetricDefinition(name="time_transfer", regex="time_transfer=([0-9\.]+)"),
    MetricDefinition(name="time_forward", regex="time_forward=([0-9\.]+)"),
    MetricDefinition(name="time_loss", regex="time_loss=([0-9\.]+)"),
    MetricDefinition(name="time_backward", regex="time_backward=([0-9\.]+)"),
    MetricDefinition(name="time_step", regex="time_step=([0-9\.]+)"),
    MetricDefinition(name="time_total", regex="time_total=([0-9\.]+)"),
    MetricDefinition(name="epoch_idx", regex="epoch_idx=([0-9\.]+)"),
    MetricDefinition(name="step_idx", regex="step_idx=([0-9\.]+)"),
]

vpc_config = {
    "subnets": ["subnet-017c79447d5151efe"],
    "security_group_ids": ["sg-0962698c0c8352ddd", "sg-03590ecb70a82d570"],
}

training_job = TrainingJob.create(
    training_job_name=job_name,
    hyper_parameters=hyper_parameters,
    algorithm_specification=AlgorithmSpecification(
        training_image=image,
        training_input_mode="File",
        metric_definitions=metric_definitions,
    ),
    role_arn=role,
    region=region,
    vpc_config=vpc_config,
    input_data_config=[
        Channel(
            channel_name="train",
            data_source=DataSource(
                file_system_data_source=FileSystemDataSource(
                    file_system_id=efs_id,
                    file_system_type="EFS",
                    file_system_access_mode="ro",
                    directory_path=efs_path,
                )
            ),
        ),
        Channel(
            channel_name="validation",
            data_source=DataSource(
                file_system_data_source=FileSystemDataSource(
                    file_system_id=efs_id,
                    file_system_type="EFS",
                    file_system_access_mode="ro",
                    directory_path=efs_path,
                )
            ),
        ),
    ],
    output_data_config=OutputDataConfig(s3_output_path=s3_output_path),
    resource_config=ResourceConfig(
        instance_type=instance_type,
        instance_count=instance_count,
        volume_size_in_gb=volume_size_in_gb,
    ),
    stopping_condition=StoppingCondition(max_runtime_in_seconds=max_runtime_in_seconds),
)

# training_job.wait(poll=20, timeout=None, logs=False)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/jose_marquez/.config/sagemaker/config.yaml


[01/22/26 21:49:14] INFO     Creating training_job resource.                                     ]8;id=659578;file:///home/jose_marquez/miniconda3/envs/traffic-gnn-v2/lib/python3.10/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=469593;file:///home/jose_marquez/miniconda3/envs/traffic-gnn-v2/lib/python3.10/site-packages/sagemaker/core/resources.py#35539\35539]8;;\

                    INFO     Runs on sagemaker prod, region:us-east-1                                  ]8;id=656950;file:///home/jose_marquez/miniconda3/envs/traffic-gnn-v2/lib/python3.10/site-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=239352;file:///home/jose_marquez/miniconda3/envs/traffic-gnn-v2/lib/python3.10/site-packages/sagemaker/core/utils/utils.py#354\354]8;;\

                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=634010;file:///home/jose_marquez/miniconda3/envs/traffic-gnn-v2/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=679975;file:///home/jose_marquez/miniconda3/envs/traffic-gnn-v2/lib/python3.10/site-packages/botocore/credentials.py#1392\1392]8;;\

In [5]:
job = TrainingJob.get(training_job_name=job_name, region=region)
print(job.training_job_status)
print(job.secondary_status)
print(job.failure_reason)

InProgress
Pending
